# 00 New Data Download for Final Model Evaluation
This notebook will strictly be used to download, save, and clean the new 2025 data that we get from the yfinance API.

## Imports

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.plotting import heatmap

import tulipy as ti

import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.decomposition import KernelPCA
from sklearn.decomposition import IncrementalPCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import initializers
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.optimizers import SGD
import keras_tuner as kt
from keras_tuner import HyperParameters

import os
import sys
import warnings

## Data Retrieval

In [10]:
# Set Scope of Date
start_date_analyzed = '2021-01-01'
end_date_analyzed = '2025-03-27'

# Retrieve S&P 500 tickers (unchanged from our previous csv as of 3/26/25)
sp500_tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
tickers = sp500_tables[0].Symbol  # Storing just the symbols into a list
tickers_changed = sp500_tables[1]  # Including the changed tickers as well to remove survivorship bias

# Getting only the added and removed tickers
tickers_added = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Added
tickers_removed = tickers_changed[pd.to_datetime(tickers_changed.Date.Date) >= start_date_analyzed].Removed

# Removing the additional added tickers
tickers = tickers[~(tickers.isin(tickers_added.Ticker))]

# Appending the removed tickers
tickers = pd.concat([tickers, tickers_removed.Ticker], ignore_index=True)

# Dropping any NA values
tickers = tickers.dropna()

# Dropping duplicates
tickers = tickers.drop_duplicates()

# Resetting the index
tickers = tickers.reset_index(drop = True)

# Filtering out problematic tickers
problematic_tickers = ['BF.B', 'BRK.B', 'GEV', 'SOLV', 'SW', 'VLTO']
filtered_tickers = [ticker for ticker in tickers if ticker not in problematic_tickers]

len(filtered_tickers)

509

In [11]:
# Getting Data for S&P500 Stocks
multi_data = yf.download(filtered_tickers, start=start_date_analyzed, end=end_date_analyzed)

# NOTE: there will be missing stocks because Yahoo Finance has delisted some of the stocks that went out of the S&P500 (like SIVB)
multi_data

[*********************100%***********************]  509 of 509 completed

28 Failed downloads:
['TWTR', 'CERN', 'DRE', 'CXO', 'VAR', 'HFC', 'PXD', 'CTXS', 'XLNX', 'PBCT', 'GPS', 'MRO', 'ALXN', 'FBHS', 'ATVI', 'ABMD', 'DISCA', 'TIF', 'DISCK', 'NLSN', 'SIVB', 'KSU', 'FRC', 'FLIR', 'MXIM', 'DISH']: YFTzMissingError('possibly delisted; no timezone found')
['CTLT']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-03-27) (Yahoo error = "No data found, symbol may be delisted")')
['TSCO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-03-27)')


Price      Adj Close                                                ...  \
Ticker          ABMD ALXN ATVI CERN CTLT CTXS CXO DISCA DISCK DISH  ...   
Date                                                                ...   
2021-01-04       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2021-01-05       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2021-01-06       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2021-01-07       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2021-01-08       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
...              ...  ...  ...  ...  ...  ...  ..   ...   ...  ...  ...   
2025-03-20       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2025-03-21       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2025-03-24       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2025-03-25       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   
2025-03-26       NaN  NaN  NaN  NaN  NaN  NaN NaN   NaN   NaN  NaN  ...   

Price      Volume                                                           \
Ticker       XLNX       XOM      XRAY       XRX      XYL      YUM      ZBH   
Date                                                                         
2021-01-04    NaN  27764700   2214500   1764900   814700  2058200  1265870   
2021-01-05    NaN  44035100   3021700   1297800  1005200  1475300   998894   
2021-01-06    NaN  36484800   1774700   2276900  1015300  1352700  1424799   
2021-01-07    NaN  29528100   1334200   1737500  1539300  2222900  1237648   
2021-01-08    NaN  28628200   1336100   1554800  1632200  1252200  1259072   
...           ...       ...       ...       ...      ...      ...      ...   
2025-03-20    NaN  16230800   2363300   3042200  1329200  1412700  1431500   
2025-03-21    NaN  41102800  12658200  34303400  2663500  3071600  2943100   
2025-03-24    NaN  14207500   2625600   4729300  2251800  1952500  1034000   
2025-03-25    NaN  12081800   2314700   5710700  1745400  1836600  1296900   
2025-03-26    NaN  14108542   2007003   5741077  1643275  1708788  1259233   

Price                                   
Ticker         ZBRA      ZION      ZTS  
Date                                    
2021-01-04   432600   1625200  1805900  
2021-01-05   374700   1349500  1358600  
2021-01-06   419500   3269500  2139800  
2021-01-07   379900   2333900  1710700  
2021-01-08   348500   2201000  1475900  
...             ...       ...      ...  
2025-03-20   499300   1874000  2026600  
2025-03-21  1471900  12259700  2886900  
2025-03-24   512000   1496200  1817400  
2025-03-25   348900   1960700  1583800  
2025-03-26   273951    950904  2078185  

[1062 rows x 2573 columns]

## Data Cleaning

In [12]:
# Dropping nulls based on column:
cleaned_multi_data = multi_data.dropna(axis=1)

cleaned_multi_data

Price            Close                                                         \
Ticker               A    AAL         AAP        AAPL        ABBV         ABT   
Date                                                                            
2021-01-04  115.582771  15.13  143.031021  126.405235   88.933952  101.045982   
2021-01-05  116.527756  15.43  142.876495  127.968094   89.853600  102.296181   
2021-01-06  119.723221  15.52  151.130753  123.660484   89.077377  102.083214   
2021-01-07  122.908958  15.38  152.421616  127.880165   90.030746  103.074104   
2021-01-08  123.785789  15.13  154.594269  128.983932   90.503227  103.361183   
...                ...    ...         ...         ...         ...         ...   
2025-03-20  120.279999  11.26   37.639999  214.100006  211.960007  126.779999   
2025-03-21  120.750000  11.39   37.459999  218.270004  210.009995  126.339996   
2025-03-24  121.970001  11.83   38.590000  220.729996  209.169998  127.209999   
2025-03-25  120.510002  11.51   37.540001  223.750000  201.339996  125.599998   
2025-03-26  119.900002  11.44   37.820000  221.529999  201.300003  126.610001   

Price                                                      ...   Volume  \
Ticker             ACN        ADBE         ADI        ADM  ...      XEL   
Date                                                       ...            
2021-01-04  242.118469  485.339996  135.426193  44.578205  ...  3197100   
2021-01-05  243.496826  485.690002  137.846848  45.445274  ...  1976800   
2021-01-06  246.159119  466.309998  138.468246  47.286682  ...  2667000   
2021-01-07  248.481583  477.739990  144.320450  46.866558  ...  2630400   
2021-01-08  249.387909  485.100006  145.368500  46.991703  ...  2221500   
...                ...         ...         ...        ...  ...      ...   
2025-03-20  300.910004  389.609985  205.899994  46.520000  ...  3039100   
2025-03-21  305.320007  387.260010  204.399994  46.099998  ...  6689200   
2025-03-24  307.179993  394.470001  212.339996  45.930000  ...  3156900   
2025-03-25  308.899994  403.640015  213.580002  46.049999  ...  3110200   
2025-03-26  311.160004  397.809998  210.529999  47.070000  ...  3074889   

Price                                                                         \
Ticker           XOM      XRAY       XRX      XYL      YUM      ZBH     ZBRA   
Date                                                                           
2021-01-04  27764700   2214500   1764900   814700  2058200  1265870   432600   
2021-01-05  44035100   3021700   1297800  1005200  1475300   998894   374700   
2021-01-06  36484800   1774700   2276900  1015300  1352700  1424799   419500   
2021-01-07  29528100   1334200   1737500  1539300  2222900  1237648   379900   
2021-01-08  28628200   1336100   1554800  1632200  1252200  1259072   348500   
...              ...       ...       ...      ...      ...      ...      ...   
2025-03-20  16230800   2363300   3042200  1329200  1412700  1431500   499300   
2025-03-21  41102800  12658200  34303400  2663500  3071600  2943100  1471900   
2025-03-24  14207500   2625600   4729300  2251800  1952500  1034000   512000   
2025-03-25  12081800   2314700   5710700  1745400  1836600  1296900   348900   
2025-03-26  14108542   2007003   5741077  1643275  1708788  1259233   273951   

Price                          
Ticker          ZION      ZTS  
Date                           
2021-01-04   1625200  1805900  
2021-01-05   1349500  1358600  
2021-01-06   3269500  2139800  
2021-01-07   2333900  1710700  
2021-01-08   2201000  1475900  
...              ...      ...  
2025-03-20   1874000  2026600  
2025-03-21  12259700  2886900  
2025-03-24   1496200  1817400  
2025-03-25   1960700  1583800  
2025-03-26    950904  2078185  

[1062 rows x 2380 columns]

In [13]:
# Making cleaned_multi_data so that there are rows for each ticker for each date:
unstacked = cleaned_multi_data.stack(level=0).unstack(level=0)
unstacked = unstacked.transpose().reset_index()
unstacked.columns.name = None

unstacked

,Ticker,Date,Close,High,Low,Open,Volume
0,A,2021-01-04,115.582771,116.995403,114.092199,115.875044,2030700.0
1,A,2021-01-05,116.527756,117.219459,115.027439,115.426878,2344000.0
2,A,2021-01-06,119.723221,120.687707,115.163820,115.631454,2428500.0
3,A,2021-01-07,122.908958,124.194943,120.911782,121.126114,1775900.0
4,A,2021-01-08,123.785789,124.350845,122.606971,123.230478,1577200.0
...,...,...,...,...,...,...,...
505507,ZTS,2025-03-20,163.160004,164.139999,162.250000,163.830002,2026600.0
505508,ZTS,2025-03-21,163.029999,163.039993,160.910004,162.880005,2886900.0
505509,ZTS,2025-03-24,163.169998,165.479996,161.699997,162.270004,1817400.0
505510,ZTS,2025-03-25,161.300003,164.350006,160.250000,163.309998,1583800.0


In [14]:
# Getting Date to datetime format and setting as index
unstacked['Date'] = pd.to_datetime(unstacked['Date'])
data=unstacked.sort_values(by="Ticker",ascending=True)
data.set_index("Date",inplace=True)

# Re-ordering as OHLCV (industry standard) format
data = data[['Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]

data.head()

,Ticker,Open,High,Low,Close,Volume
Date,,,,,,
2021-01-04,A,115.875044,116.995403,114.092199,115.582771,2030700.0
2023-10-13,A,107.094515,109.662328,105.904799,109.563187,2807600.0
2023-10-16,A,110.088643,111.823649,108.700636,110.544701,1675500.0
2023-10-17,A,109.156692,111.942621,108.274317,111.436989,1682800.0
2023-10-18,A,110.376156,110.376156,107.610059,107.679459,1803500.0


In [15]:
# Creating a new Date_col column to rank and drop dates appropriately
data = data.reset_index().set_index("Date", drop=False)
data = data.rename(columns={"Date": "Date_col"})
data["rank"] = data.groupby("Ticker")["Date_col"].rank()
data = data.sort_values(by=["Ticker", "Date_col"])

data

,Date_col,Ticker,Open,High,Low,Close,Volume,rank
Date,,,,,,,,
2021-01-04,2021-01-04,A,115.875044,116.995403,114.092199,115.582771,2030700.0,1.0
2021-01-05,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0
2021-01-06,2021-01-06,A,115.631454,120.687707,115.163820,119.723221,2428500.0,3.0
2021-01-07,2021-01-07,A,121.126114,124.194943,120.911782,122.908958,1775900.0,4.0
2021-01-08,2021-01-08,A,123.230478,124.350845,122.606971,123.785789,1577200.0,5.0
...,...,...,...,...,...,...,...,...
2025-03-20,2025-03-20,ZTS,163.830002,164.139999,162.250000,163.160004,2026600.0,1058.0
2025-03-21,2025-03-21,ZTS,162.880005,163.039993,160.910004,163.029999,2886900.0,1059.0
2025-03-24,2025-03-24,ZTS,162.270004,165.479996,161.699997,163.169998,1817400.0,1060.0


## Comparing with original ticker data (2021-2024)

In [17]:
# Loading in stock data
data_orig = pd.read_csv("../SP500_4y_day_data.csv", index_col='Date', parse_dates=True)

# Converting Date_col to datetime format
data_orig['Date_col'] = pd.to_datetime(data_orig['Date_col'])

data_orig

,Date_col,Ticker,Open,High,Low,Close,Volume,rank
Date,,,,,,,,
2021-01-04,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0
2021-01-05,2021-01-05,A,115.426870,117.219451,115.027432,116.527748,2344000.0,2.0
2021-01-06,2021-01-06,A,115.631454,120.687707,115.163820,119.723221,2428500.0,3.0
2021-01-07,2021-01-07,A,121.126122,124.194950,120.911790,122.908966,1775900.0,4.0
2021-01-08,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0
...,...,...,...,...,...,...,...,...
2024-12-24,2024-12-24,ZTS,163.206597,164.542554,162.249499,164.203583,1023600.0,1001.0
2024-12-26,2024-12-26,ZTS,163.226540,165.290293,162.528653,165.021118,2167200.0,1002.0
2024-12-27,2024-12-27,ZTS,164.452833,166.028073,163.027150,164.103897,1800100.0,1003.0


While there are some slight differences, the data looks pretty similar overall. We can also check to make sure all the tickers in this dataset is also in our previous dataset.

In [35]:
# Getting the unique tickers for both datasets
data_tickers = data['Ticker'].unique()
data_orig_tickers = data_orig['Ticker'].unique()

for ticker in data_tickers:
    if ticker not in data_orig_tickers:
        print("Ticker", ticker, "is not in the original dataset.")
print("")

for ticker in data_orig_tickers:
    if ticker not in data_tickers:
        print("Ticker", ticker, "is new in this dataset.")

Ticker ETR is not in the original dataset.
Ticker LEN is not in the original dataset.

Ticker TSCO is new in this dataset.
Ticker WBD is new in this dataset.


We now see that we have two tickers that were in the original data that - for some reason or another - was not downloaded from the yfinance API; conversely, there appear to be two new tickers as well. Looking at each individual ticker, it's unclear why this was the case. However, because all of these tickers have been in the S&P500 at one point or another, we will assume our model can learn and generalize from these tickers as well, and we will keep this new dataset consistent.

In [38]:
# Downloading the tickers into an updated csv file
data.to_csv('../SP500_updated_data_2025_03_26.csv')

In [39]:
# Getting the tickers back just to confirm
data_updated_2025_03_26 = pd.read_csv("../SP500_updated_data_2025_03_26.csv", index_col='Date', parse_dates=True)

# Converting Date_col to datetime format
data_updated_2025_03_26['Date_col'] = pd.to_datetime(data_updated_2025_03_26['Date_col'])

data_updated_2025_03_26

,Date_col,Ticker,Open,High,Low,Close,Volume,rank
Date,,,,,,,,
2021-01-04,2021-01-04,A,115.875044,116.995403,114.092199,115.582771,2030700.0,1.0
2021-01-05,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0
2021-01-06,2021-01-06,A,115.631454,120.687707,115.163820,119.723221,2428500.0,3.0
2021-01-07,2021-01-07,A,121.126114,124.194943,120.911782,122.908958,1775900.0,4.0
2021-01-08,2021-01-08,A,123.230478,124.350845,122.606971,123.785789,1577200.0,5.0
...,...,...,...,...,...,...,...,...
2025-03-20,2025-03-20,ZTS,163.830002,164.139999,162.250000,163.160004,2026600.0,1058.0
2025-03-21,2025-03-21,ZTS,162.880005,163.039993,160.910004,163.029999,2886900.0,1059.0
2025-03-24,2025-03-24,ZTS,162.270004,165.479996,161.699997,163.169998,1817400.0,1060.0
